In [17]:
import torch
from torch.serialization import add_safe_globals  # 新增：用于白名单机制
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# 1. 重新定义模型结构（必须与训练时完全一致）
class LSTMModel(torch.nn.Module):
    def __init__(self, input_size=144, hidden_size=50, num_layers=1, output_size=60):
        super(LSTMModel, self).__init__()
        self.lstm = torch.nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = torch.nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])  # 只取最后一个时间步的输出
        return out

In [18]:
# 3. 加载模型
try:
    model = torch.load("lstm_144_60.pth", map_location="cpu", weights_only=False)  # 关键：禁用 weights_only
    model.eval()  # 设置为评估模式
except Exception as e:
    raise RuntimeError(f"模型加载失败: {str(e)}")

In [20]:
# 获取输入输出的特征维度
input_features = model.lstm.input_size  # 输入特征数：144
output_features = model.fc.out_features  # 输出特征数：1
seq_length = 10  # 根据训练时的序列长度设置（需已知或调整）

print(f"输入形状: (batch_size, sequence_length, input_features) = (N, {seq_length}, {input_features})")
print(f"输出形状: (batch_size, output_features) = (N, {output_features})")

输入形状: (batch_size, sequence_length, input_features) = (N, 10, 1)
输出形状: (batch_size, output_features) = (N, 60)


In [5]:
# 生成随机测试数据
batch_size = 1
test_input = torch.randn(batch_size, seq_length, input_features)

# 前向传播
with torch.no_grad():
    test_output = model(test_input)

# 打印结果
print("\n测试输入形状:", test_input.shape)
print("测试输出形状:", test_output.shape)


测试输入形状: torch.Size([1, 10, 1])
测试输出形状: torch.Size([1, 60])


In [27]:
# 测试数据
input_sequence = [round(i*0.1, 1) for i in range(1, 145)]  # 保留1位小数
print(input_sequence)

[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0, 2.1, 2.2, 2.3, 2.4, 2.5, 2.6, 2.7, 2.8, 2.9, 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 4.0, 4.1, 4.2, 4.3, 4.4, 4.5, 4.6, 4.7, 4.8, 4.9, 5.0, 5.1, 5.2, 5.3, 5.4, 5.5, 5.6, 5.7, 5.8, 5.9, 6.0, 6.1, 6.2, 6.3, 6.4, 6.5, 6.6, 6.7, 6.8, 6.9, 7.0, 7.1, 7.2, 7.3, 7.4, 7.5, 7.6, 7.7, 7.8, 7.9, 8.0, 8.1, 8.2, 8.3, 8.4, 8.5, 8.6, 8.7, 8.8, 8.9, 9.0, 9.1, 9.2, 9.3, 9.4, 9.5, 9.6, 9.7, 9.8, 9.9, 10.0, 10.1, 10.2, 10.3, 10.4, 10.5, 10.6, 10.7, 10.8, 10.9, 11.0, 11.1, 11.2, 11.3, 11.4, 11.5, 11.6, 11.7, 11.8, 11.9, 12.0, 12.1, 12.2, 12.3, 12.4, 12.5, 12.6, 12.7, 12.8, 12.9, 13.0, 13.1, 13.2, 13.3, 13.4, 13.5, 13.6, 13.7, 13.8, 13.9, 14.0, 14.1, 14.2, 14.3, 14.4]


In [26]:
import requests
import json

# 测试数据
input_sequence = [round(i*0.1, 1) for i in range(1, 145)]  # 保留1位小数

# 发送请求
response = requests.post(
    "http://localhost:8001/predict",
    json={"input_sequence": input_sequence}
)

# 解析结果
if response.status_code == 200:
    print("✅ 服务运行正常")
    result = response.json()
    print(f"输入长度: {len(result['input_sequence'])}")
    print(f"预测数量: {result['prediction_count']}")
    print("前5个预测值示例:", [round(x, 4) for x in result['predicted_values'][:5]])
else:
    print("❌ 服务异常:", response.text)

✅ 服务运行正常
输入长度: 144
预测数量: 60
前5个预测值示例: [6.5275, 10.5696, 10.8275, 10.843, 10.852]


In [29]:
result.keys()

dict_keys(['input_sequence', 'predicted_values', 'prediction_count'])

In [31]:
result['prediction_count']

60

In [32]:
result['predicted_values']

[6.527453422546387,
 10.569607734680176,
 10.827467918395996,
 10.842957496643066,
 10.851982116699219,
 10.806340217590332,
 11.64268684387207,
 11.80889892578125,
 12.136435508728027,
 11.203779220581055,
 11.361475944519043,
 11.595771789550781,
 11.074419021606445,
 11.426641464233398,
 11.144386291503906,
 11.299787521362305,
 11.51424789428711,
 11.152179718017578,
 11.344826698303223,
 10.945509910583496,
 11.09468936920166,
 11.451961517333984,
 11.303112983703613,
 11.444045066833496,
 11.176281929016113,
 11.784399032592773,
 11.386085510253906,
 11.523880004882812,
 11.064007759094238,
 11.132386207580566,
 10.882528305053711,
 11.090176582336426,
 11.326360702514648,
 11.114115715026855,
 11.356522560119629,
 11.170273780822754,
 10.991808891296387,
 11.225177764892578,
 11.178950309753418,
 11.080140113830566,
 10.727784156799316,
 10.803081512451172,
 10.846244812011719,
 11.127037048339844,
 11.020147323608398,
 10.817486763000488,
 10.968855857849121,
 11.19344711303711

In [36]:
from datetime import datetime, timedelta
import requests

# 获取最近3小时的数据
end_time = datetime.now()
start_time = end_time - timedelta(hours=1)

# 构建API请求参数
params = {
    "startTime": start_time.isoformat(timespec='milliseconds'),
    "endTime": end_time.isoformat(timespec='milliseconds'),
    "interval": 60000,   # 1分钟间隔
    "valueonly": 0,
    "decimal": 2,
    "names": "DLDZ_DQ200_LLJ01_FQ01.PV,DLDZ_AVS_LLJ01_FQ01.PV"
}

# 存储差分和结果
sum_diffs = []

# 发送GET请求

response = requests.get(
    "http://8.130.25.118:8000/api/hisdata",
    params=params,
    timeout=10
)

if response.status_code == 200:
    data = response.json()

In [37]:
# 解析响应
data = response.json()
if data['code'] != 0:
    raise ValueError(f"API返回错误码: {data['code']}")

In [ ]:
data

{'code': 0,
 'items': [{'name': 'DLDZ_DQ200_LLJ01_FQ01.PV',
   'vals': [{'time': '2025-05-27T09:39:07.237', 'val': 334724256.0},
    {'time': '2025-05-27T09:40:07.237', 'val': 334724416.0},
    {'time': '2025-05-27T09:41:07.237', 'val': 334724608.0},
    {'time': '2025-05-27T09:42:07.237', 'val': 334724800.0},
    {'time': '2025-05-27T09:43:07.237', 'val': 334724960.0},
    {'time': '2025-05-27T09:44:07.237', 'val': 334725152.0},
    {'time': '2025-05-27T09:45:07.237', 'val': 334725344.0},
    {'time': '2025-05-27T09:46:07.237', 'val': 334725536.0},
    {'time': '2025-05-27T09:47:07.237', 'val': 334725696.0},
    {'time': '2025-05-27T09:48:07.237', 'val': 334725888.0},
    {'time': '2025-05-27T09:49:07.237', 'val': 334726080.0},
    {'time': '2025-05-27T09:50:07.237', 'val': 334726240.0},
    {'time': '2025-05-27T09:51:07.237', 'val': 334726432.0},
    {'time': '2025-05-27T09:52:07.237', 'val': 334726592.0},
    {'time': '2025-05-27T09:53:07.237', 'val': 334726784.0},
    {'time': '202

In [40]:
from datetime import datetime, timedelta
import requests

# 获取最近3小时的数据
end_time = datetime.now()
start_time = end_time - timedelta(hours=3)

# 构建API请求参数
params = {
    "startTime": start_time.isoformat(timespec='milliseconds'),
    "endTime": end_time.isoformat(timespec='milliseconds'),
    "interval": 60000,   # 1分钟间隔
    "valueonly": 0,
    "decimal": 2,
    "names": "DLDZ_DQ200_LLJ01_FQ01.PV,DLDZ_AVS_LLJ01_FQ01.PV"
}

# 存储差分和结果
sum_diffs = []

# 发送GET请求
try:
    response = requests.get(
        "http://8.130.25.118:8000/api/hisdata",
        params=params,
        timeout=10
    )
    response.raise_for_status()
    
    # 解析API响应数据
    api_data = response.json()
    
    # 检查API状态码
    if api_data.get("code") != 0:
        raise Exception(f"API返回错误码: {api_data.get('code')}")
    
    items = api_data.get("items", [])
    
    # 初始化字段值列表
    field1_vals = []
    field2_vals = []
    
    # 提取两个字段的数值序列
    for item in items:
        name = item.get("name")
        vals = item.get("vals", [])
        
        if not vals:
            raise Exception(f"字段 {name} 返回空数据")
            
        # 按时间戳排序（防止单个字段数据乱序）
        sorted_vals = sorted(vals, key=lambda x: x["time"])
        
        if name == "DLDZ_DQ200_LLJ01_FQ01.PV":
            field1_vals = [v["val"] for v in sorted_vals]
        elif name == "DLDZ_AVS_LLJ01_FQ01.PV":
            field2_vals = [v["val"] for v in sorted_vals]
    
    # 检查数据完整性
    if len(field1_vals) != len(field2_vals):
        raise Exception(f"字段长度不一致：{len(field1_vals)} vs {len(field2_vals)}")
    
    # 计算差分并求和
    for i in range(1, len(field1_vals)):
        diff1 = field1_vals[i] - field1_vals[i-1]
        diff2 = field2_vals[i] - field2_vals[i-1]
        sum_diffs.append(diff1 + diff2)
        
except Exception as e:
    raise Exception(f"处理失败: {str(e)}")

In [ ]:
print(type(sum_diffs))
sum_diffs[:5]  
print(len(sum_diffs)) # 差分之后是179个，少一个数

<class 'list'>
179
